# GEOG 464 Project pipeline: CSV to GeoJSON

This notebook is the data link of the course pipeline: **CSV -> Python -> GeoJSON -> Leaflet**.

```
geog464-starter/
  data/sample_places.csv     input table (edit this)
  data/places.geojson        output, written by this notebook
  notebooks/mvp_pipeline.ipynb
  site/index.html, style.css, app.js
```

Run each cell in order, top to bottom. Cell 3 writes `data/places.geojson`, which the web page in `site/` draws. If a cell fails with NameError, run the cells above it first (or use Run All).


**1) Importing libraries and loading the files (For the GeoJSON)**  

Installing the kernal to be able to load the geopandas make sure you are on python 3.13.16 for the kernals 

In [4]:
%pip install geopandas 


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Testing to see if the GEOpandas has imported correctly and actually works 

In [5]:
import geopandas as gpd

print("GeoPandas is working")
#test to see if geopandas is working

GeoPandas is working


Actually importing the data from the GeoJson and storing it

In [6]:
import geopandas as gpd

data = gpd.read_file("../Data/Aires_repartition_MT.geojson")

data.head()

,DESC_ENTIT,PRODUCTEUR,NOM_FRANCA,NOM_ANGLA,NOM_SCIENT,FAMILLE,GRAND_GROUPE,DATE_MAJ,Shape_Length,Shape_Area,geometry
0,Aire de répartition,MELCCFP,Belette à longue queue,Long-tailed weasel,Mustela frenata,Mustelidae,Mammifères,2021,6.801074e+06,3.436918e+11,"MULTIPOLYGON (((-485026.718 132619.355, -48503..."
1,Aire de répartition,MELCCFP,Belette pygmée,Least weasel,Mustela nivalis,Mustelidae,Mammifères,2021,1.963463e+07,1.459568e+12,"MULTIPOLYGON (((-485026.718 132619.355, -48503..."
2,Aire de répartition,MELCCFP,Boeuf musqué,Muskox,Ovibos moschatus,Bovidae,Mammifères,2021,8.181692e+06,5.452414e+11,"MULTIPOLYGON (((-569019.31 1304248.092, -56770..."
3,Aire de répartition,MELCCFP,Campagnol à dos roux de Gapper,Southern red-backed vole,Myodes gapperi,Cricetidae,Mammifères,2021,1.641556e+07,1.403891e+12,"MULTIPOLYGON (((-485026.718 132619.355, -48504..."
4,Aire de répartition,MELCCFP,Campagnol des champs,Meadow Vole,Microtus pennsylvanicus,Cricetidae,Mammifères,2021,1.847168e+07,1.534596e+12,"MULTIPOLYGON (((-485026.718 132619.355, -48504..."


Looking at the data size and shape, column names as well as the ordiante systems 

In [8]:
data.shape

(69, 11)

In [ ]:
list(data.columns)

In [ ]:
data.geometry.head()

## 2) (Optional) Quick filtering/cleaning
For MVP, just loading is enough. Example filter: keep only rows whose `category` is `Landmark`. This cell only displays a filtered copy; it does not change `df`, so the export below still writes every row. Lab 2 is where filtering becomes yours.

This gives e the names and amount of features in the Data set 

In [ ]:
features = data.get("features", [])

print("Total features:", len(features))

print(features[0]["properties"].keys())



Total features: 69
dict_keys(['DESC_ENTIT', 'PRODUCTEUR', 'NOM_FRANCA', 'NOM_ANGLA', 'NOM_SCIENT', 'FAMILLE', 'GRAND_GROUPE', 'DATE_MAJ', 'Shape_Length', 'Shape_Area'])


Checking to see if any features have blank or missing information

In [15]:

features = data.get("features", [])

# Find all the property fields in the dataset
all_fields = set()

for feature in features:
    properties = feature.get("properties") or {}
    all_fields.update(properties.keys())

# Count missing or blank values in each field
for field in sorted(all_fields):
    blank_count = 0

    for feature in features:
        properties = feature.get("properties") or {}
        value = properties.get(field)

        if value is None or (
            isinstance(value, str) and value.strip() == ""
        ):
            blank_count += 1

    print(f"{field}: {blank_count} blank or missing values")


DATE_MAJ: 0 blank or missing values
DESC_ENTIT: 0 blank or missing values
FAMILLE: 0 blank or missing values
GRAND_GROUPE: 0 blank or missing values
NOM_ANGLA: 0 blank or missing values
NOM_FRANCA: 0 blank or missing values
NOM_SCIENT: 0 blank or missing values
PRODUCTEUR: 0 blank or missing values
Shape_Area: 0 blank or missing values
Shape_Length: 0 blank or missing values


Checking for inconsistenses in naming and formation ( capitalizations/ spaces) as well as any repeated species or names 

In [16]:

from collections import Counter

for field in ["NOM_FRANCA", "NOM_ANGLA"]:
    names = []

    for feature in features:
        properties = feature.get("properties") or {}
        value = properties.get(field)

        if value is not None and str(value).strip() != "":
            names.append(str(value).strip())

    print(f"\n--- {field} ---")
    print("Total nonblank entries:", len(names))
    print("Unique names:", len(set(names)))

    print("\nMost common names:")
    for name, count in Counter(names).most_common(20):
        print(f"{name}: {count}")



--- NOM_FRANCA ---
Total nonblank entries: 69
Unique names: 69

Most common names:
Belette à longue queue: 1
Belette pygmée: 1
Boeuf musqué: 1
Campagnol à dos roux de Gapper: 1
Campagnol des champs: 1
Campagnol des rochers: 1
Campagnol sylvestre: 1
Campagnol-lemming boréal: 1
Campagnol-lemming de Cooper: 1
Carcajou: 1
Caribou: 1
Castor du Canada: 1
Cerf de Virginie: 1
Chauve-souris argentée: 1
Chauve-souris cendrée: 1
Chauve-souris nordique: 1
Chauve-souris pygmée de l'Est: 1
Chauve-souris rousse: 1
Condylure à nez étoilé: 1
Coyote: 1

--- NOM_ANGLA ---
Total nonblank entries: 69
Unique names: 69

Most common names:
Long-tailed weasel: 1
Least weasel: 1
Muskox: 1
Southern red-backed vole: 1
Meadow Vole: 1
Rock Vole: 1
Woodland Vole: 1
Northern bog Lemming: 1
Southern bog Lemming: 1
Wolverine: 1
Woodland caribou: 1
American beaver: 1
White-tailed deer: 1
Silver-haired bat: 1
Hoary bat: 1
Northern long-eared bat: 1
Eastern small-footed bat: 1
Red bat: 1
Star-nosed Mole: 1
coyote: 1


In [17]:

for field in ["NOM_FRANCA", "NOM_ANGLA", "NOM_SCIENT"]:
    variants = {}

    for feature in features:
        properties = feature.get("properties") or {}
        value = properties.get(field)

        if value is None or str(value).strip() == "":
            continue

        original = str(value)
        normalized = original.strip().casefold()

        variants.setdefault(normalized, set()).add(original)

    print(f"\n--- Possible formatting inconsistencies: {field} ---")

    found = False
    for normalized, spellings in variants.items():
        if len(spellings) > 1:
            print(sorted(spellings))
            found = True

    if not found:
        print("No capitalization or whitespace variants found.")



--- Possible formatting inconsistencies: NOM_FRANCA ---
No capitalization or whitespace variants found.

--- Possible formatting inconsistencies: NOM_ANGLA ---
No capitalization or whitespace variants found.

--- Possible formatting inconsistencies: NOM_SCIENT ---
No capitalization or whitespace variants found.


After all these steps I can see there are some features that I do not need such as DESC_ENTIT", "PRODUCTEUR", "GRAND_GROUPE" as they wont be helpful information on the map and just create a lot more to load 

In [24]:

# Fields that we don't need for the web map
fields_to_remove = [
    "DESC_ENTIT",
    "PRODUCTEUR",
    "GRAND_GROUPE",
    "Date_maj"
]

# Remove these fields from every feature's properties
for feature in data["features"]:
    for field in fields_to_remove:
        feature["properties"].pop(field, None)


Save a new file with the cleaned version of the information 

In [25]:

import json

with open("../Data/Aires_repartition_MT_cleaned.geojson", "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False)

print("File saved in the data folder!")


File saved in the data folder!


In [22]:

print(data["features"][0]["properties"].keys())


dict_keys(['NOM_FRANCA', 'NOM_ANGLA', 'NOM_SCIENT', 'FAMILLE', 'DATE_MAJ', 'Shape_Length', 'Shape_Area'])


In [28]:
import json
import copy

cleaned_data = copy.deepcopy(data)

fields_to_keep = [
    "NOM_FRANCA",
    "NOM_ANGLA",
    "NOM_SCIENT",
    "FAMILLE",
    "Shape_Length",
    "Shape_Area"
]

for feature in cleaned_data["features"]:
    properties = feature["properties"]

    feature["properties"] = {
        field: properties[field]
        for field in fields_to_keep
        if field in properties
    }

with open("../data/habitats_cleaned.geojson", "w", encoding="utf-8") as file:
    json.dump(cleaned_data, file, ensure_ascii=False, indent=2)

## 3) Convert DataFrame to GeoJSON
We’ll iterate through rows and build a GeoJSON `FeatureCollection`. **Output:** `../data/places.geojson`.   
**DO NOT RUN THIS FOR THE HABITS FILE** 

In [ ]:
import json
features = []
for _, row in df.iterrows():
    try:
        lat = float(row["lat"])
        lon = float(row["lon"])
    except Exception:
        continue  # skip bad rows
    feature = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lon, lat]},
        "properties": {
            "name": str(row.get("name", "")).strip(),
            "category": str(row.get("category", "")).strip(),
            "description": str(row.get("description", "")).strip()
        }
    }
    features.append(feature)
geojson = {"type": "FeatureCollection", "features": features}
out_path = "../data/places.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)
print(f"GeoJSON created with {len(features)} features, written to {out_path}")

## 4) Preview the GeoJSON
Read the file back in and show the first feature.

In [29]:
with open("../data/habitats_cleaned.geojson", "r", encoding="utf-8") as f:
    data = json.load(f)
print("Features:", len(data.get("features", [])))
data["features"][0] if data.get("features") else {}

Features: 69


{'type': 'Feature',
 'geometry': {'type': 'MultiPolygon',
  'coordinates': [[[[-485026.7177999988, 132619.3553999998],
     [-485039.2480999976, 132623.0590000004],
     [-485041.3927000016, 132623.6928999983],
     [-485063.8632000014, 132632.3627999984],
     [-485084.0806000009, 132644.8630000018],
     [-485089.6335000023, 132651.2936000004],
     [-485089.4579999968, 132653.7699000016],
     [-485088.7123000026, 132664.2930999994],
     [-485093.1950000003, 132671.6376000009],
     [-485104.757600002, 132678.5020999983],
     [-485106.8554999977, 132685.2221000008],
     [-485151.4980000034, 132721.9034000002],
     [-485153.8034000024, 132723.2197000012],
     [-485156.1463000029, 132725.7131999992],
     [-485195.0653999969, 132757.7098000012],
     [-485205.3422999978, 132761.0821999982],
     [-485238.2414999977, 132764.4470000006],
     [-485255.8134000003, 132771.7454999983],
     [-485299.0680000037, 132798.9743999988],
     [-485309.5863000005, 132810.3603000008],
     [-4